# Fix: Mobility vs Aggression Scatter Plot

The original `scramble_rate_mobility` clustered everyone near zero because it only
counted scrambles on pass plays. This fix incorporates designed QB runs, rushing
yards per game, and rushing EPA to build a proper mobility score.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
# Run the mobility fix script first (only need to do this once)
!cd .. && PYTHONPATH=. python fix_mobility.py

In [ ]:
# Load the improved mobility data and merge with existing features
features = pd.read_parquet('../data/processed/qb_features.parquet')
mobility = pd.read_parquet('../data/processed/qb_mobility.parquet')

# Merge the new mobility columns into features
new_cols = ['mobility_score', 'rush_yards_per_game', 'designed_rush_rate', 'total_mobility_rate']
for col in new_cols:
    if col in mobility.columns:
        features[col] = mobility[col]

features = features.fillna(0)
plot_df = features.reset_index()
plot_df.head()

In [ ]:
# FIXED: Mobility vs Aggression — now with real spread
fig = px.scatter(
    plot_df, 
    x='mobility_score', 
    y='avg_air_yards',
    text='player_name', 
    size='pass_attempts',
    color='clutch_epa',
    color_continuous_scale='RdYlGn',
    title='Mobility vs Aggression (colored by Clutch EPA)',
    labels={
        'mobility_score': 'Mobility Score (scrambles + designed runs + rush yards)',
        'avg_air_yards': 'Average Air Yards per Attempt',
        'clutch_epa': 'Clutch EPA'
    },
    hover_data=['team', 'rush_yards_per_game', 'designed_rush_rate']
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(
    width=1000, height=700,
    plot_bgcolor='white',
)

# Add quadrant lines at the median
fig.add_hline(y=plot_df['avg_air_yards'].median(), line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=plot_df['mobility_score'].median(), line_dash='dash', line_color='gray', opacity=0.5)

# Add quadrant labels
fig.add_annotation(x=plot_df['mobility_score'].max()*0.85, y=plot_df['avg_air_yards'].max()*0.98,
                   text='Mobile Gunslingers', showarrow=False, font=dict(size=11, color='gray'))
fig.add_annotation(x=plot_df['mobility_score'].min()*0.85, y=plot_df['avg_air_yards'].max()*0.98,
                   text='Pocket Gunslingers', showarrow=False, font=dict(size=11, color='gray'))
fig.add_annotation(x=plot_df['mobility_score'].max()*0.85, y=plot_df['avg_air_yards'].min()*1.02,
                   text='Mobile Game Managers', showarrow=False, font=dict(size=11, color='gray'))
fig.add_annotation(x=plot_df['mobility_score'].min()*0.85, y=plot_df['avg_air_yards'].min()*1.02,
                   text='Pocket Game Managers', showarrow=False, font=dict(size=11, color='gray'))

fig.show()

In [ ]:
# Alternative view: Rush Yards/Game vs Air Yards — very intuitive
fig2 = px.scatter(
    plot_df,
    x='rush_yards_per_game',
    y='avg_air_yards',
    text='player_name',
    size='pass_attempts',
    color='avg_intended_epa',
    color_continuous_scale='RdYlGn',
    title='Rushing Production vs Downfield Aggression (colored by EPA)',
    labels={
        'rush_yards_per_game': 'Rush Yards per Game',
        'avg_air_yards': 'Average Air Yards per Attempt',
        'avg_intended_epa': 'EPA/Play'
    },
    hover_data=['team']
)
fig2.update_traces(textposition='top center', textfont_size=9)
fig2.update_layout(width=1000, height=700, plot_bgcolor='white')
fig2.show()

In [ ]:
# Save updated features with new mobility columns
features.to_parquet('../data/processed/qb_features.parquet')
print('Updated features saved with new mobility metrics!')